### Module Loading

In [ ]:
import Pkg
Pkg.activate("..")


In [ ]:

using Plots,JLD2,Statistics,Measures,LaTeXStrings,PGFPlotsX,Random,LinearAlgebra


In [ ]:
push!(PGFPlotsX.CUSTOM_PREAMBLE,
      "\\usepackage{amsmath}\n\\usepackage{amssymb}")

In [ ]:
r_tilde_poisson = 2*log(2)-1;
r_tilde_GOE=4-2*sqrt(3); 
r_tilde_GUE=2*(sqrt(3)/pi)- 0.5;
r_tilde_GSE=(32/15)*(sqrt(3)/pi)-0.5;

## Plots

#### Interaction Strength

Interaction profile of the clean HS gen Hamiltonian for $N{=}22$ sites, i.e., the interaction strength between a given site and its neighbors as a function of distance ($r$) for different values of the interaction parameter $\alpha$. The circles indicate the distances from the successive neighboring sites.

In [ ]:
pgfplotsx()

guide_fs  = 3 * 20
tick_fs   = 3 * 20
legend_fs = 3 * 12
title_fs  = 3 * 15

# Parameters
N=22
r=0.01:0.01:2.0
alpha_list = [0.0,0.2,0.4,1.0,1.6,2.0,3.0,100]
yli=0.51

# Colormap for alpha
palette = cgrad(:tab10)

#defining Kac normalization factor
function kac_fac(alpha,N)
    kac=0
    d=[exp(im*(2*pi/N)*i) for i in 1:N]
    for i in 1:N
        for j in 1:N
            if i==j
                kac+=0
            else
                kac += ((2 * pi / N) * (1/abs(d[i] - d[j])))^alpha
            end
        end
    end
    return kac/(N)
end

# Marker list

markers = [:rect, :circle,  :pentagon,:diamond, :utriangle, :dtriangle,:ltriangle,:rtriangle,]
#Plot setup
p1 = plot(
    xlabel = L" \mathrm{Distance\,} r",
    ylabel = L" \mathrm{Interaction\,\, strength}",
    legend = :topright,
    legend_font_halign = :left,
    legend_columns = 2,
    framestyle = :box,
    size = (800, 600),
    dpi = 300,
    grid = false,
    guidefontsize = guide_fs,
    tickfontsize = tick_fs,
    legendfontsize = legend_fs,
    titlefontsize = title_fs,
    left_margin = 7mm,
    right_margin = 20mm,       
    bottom_margin = 7mm,
    fontfamily = "Computer Modern",
    ylim=(0,yli)
)


for (i,a) in enumerate(alpha_list)
    raw = ((2*pi/N)./r).^a / kac_fac(a,N)
    y_vals = clamp.(raw, 0, yli)

    #ploting the interaction strength curve
    plot!(p1, r, y_vals;
        linewidth = 1.2,
        color = palette[i],
        label="")

    int_sites = 1:floor(Int,N/2-1)  
    int_vals  = ((pi/N)./sin.((pi/N).*int_sites)).^a /kac_fac(a,N)
    int_r = 2*sin.((pi/N).*int_sites)
    int_mask=clamp.(int_vals, 0, yli)

    #plotting the lattice sites
    scatter!(p1, int_r, int_mask;
        marker = markers[i],
        ms = 5,          
        color = palette[i],
        label = "") 
        
    plot!(p1, [1], [-1];
    linewidth = 1.5,
    color = palette[i],
    marker = markers[i],
    ms = 5,
    label = L"\alpha = %$a")
end

display(p1)


In [ ]:
savefig(p1,"figures/Interaction_strength_vs_distance_N_$(N).pdf")

#### Clean Plots

Mean gap ratio $\langle \tilde{\mathfrak{r}}\rangle$ of the clean Hamiltonian as a function of $\alpha$, where $1/r^{\alpha}$ is the interaction between any two spins separated by a chord distance of $r$, for $N{=}22$ spin-$1/2$ particles on a ring in the $(m,k){=}(0,1)$ symmetry sector, where $m$ is the magnetization and $k$ is the pseudo-momentum [To avoid commutation with parity, results for the second-largest sector with $m{=}0$, i.e., $k{=}1$ are shown]. Also shown, are results of disordered average gap ratio $\langle \tilde{\mathfrak{r}}\rangle$ for the position-disordered Hamiltonian, where $\delta$ parametrizes the strength of the position-disorder, for $N{=}16$ spins in the $m{=}0$ sector.

In [ ]:
file1="r_stat_HS_gen.jld2"
file2="r_stat_HS_gen_std_error.jld2"
@load file1 r_stat_HS_gen;
@load file2 r_stat_HS_gen_std_error;

In [ ]:
pgfplotsx()  

# Font sizes
guide_fs  = 5 * 7
tick_fs   = 5 * 20
legend_fs = 5 * 12
title_fs  = 5 * 15

# Parameters
delta_list = [0.0, 0.1,0.3, 0.5, 0.8,1.0]
hjj = 0.0
mz = 0.0
alpha_list=0.0:0.2:3.0
alpha_zoom = [0.01,0.05,0.1]

# Reference lines
hline_values = [
    (r_tilde_GOE,     :darkorange, :dot,  L"\mathrm{GOE}"),
    (r_tilde_poisson, :green, :dash,  L"\mathrm{Poisson}"),
]

vline_values = [
    (0.0, :gray, :dash),
    (2.0, :gray, :dash),
    (1.0, :gray, :dash)
]

# Main plot setup
p0 = plot(
    xlabel = L"\alpha",
    ylabel = L"{\small \langle \tilde{\mathfrak{r}} \rangle}",
    title  = L"h/J_\alpha^{\mathrm{NN}}{=}%$hjj",
    legend =(0.2,0.35),
    legend_columns=2,
    framestyle = :box,
    size = (800, 600),
    dpi = 300,
    grid = false,
    guidefontsize = guide_fs,
    tickfontsize = tick_fs,
    legendfontsize = legend_fs,
    titlefontsize = title_fs,
    left_margin = 7mm,
    right_margin = 5mm,
    bottom_margin = 7mm,
    fontfamily = "Computer Modern",
    xticks = collect(floor(Int, minimum(alpha_list)):ceil(Int, maximum(alpha_list))),
)

# Add horizontal reference lines
for (y, col, style, lbl) in hline_values
    hline!(p0, [y]; color=col, linestyle=style,label="", linewidth=4, alpha=1.0)
end

# Add vertical reference lines
for (x, col, style) in vline_values
    vline!(p0, [x]; color=col, linestyle=style, label="", linewidth=4, alpha=1.5)
end

# Markers and colors
markers = [:rect, :circle, :pentagon, :diamond, :utriangle, :dtriangle]
marker_size=[12,10,9,9,9,9]

colors = [
    "#E69F00",  # orange → square
    "#009E73",  # green → circle
     "#CC79A7",  # reddish purple → hexagon
    "#D55E00",  # vermillion → diamond
    "#F0E442",  # yellow → up-triangle
    "#56B4E9",  # sky blue → down-triangle
]

# Loop over delta values
for (i, delta) in enumerate(delta_list)
    if delta==0.0
        
        N=22
        alpha_list=vcat(alpha_zoom,0.0:0.2:3.0)
    else
        N=16
        alpha_list=vcat(alpha_zoom,0.2:0.2:3.0)
    end
    
    # Mean ⟨r̃⟩ values
    r_mean_avg_list = [
        r_stat_HS_gen[(N, a, mz, delta, hjj)]
        for a in alpha_list
    ]

    # Corresponding standard errors
    r_std_err_list = [
        r_stat_HS_gen_std_error[(N, a, mz, delta, hjj)]
        for a in alpha_list
    ]

    # Scatter plot with vertical error bars
    scatter!(p0,alpha_list,r_mean_avg_list;
    yerror = r_std_err_list,
    label = L"\delta {=} %$delta\,\,",
    marker = markers[i],
    markersize = marker_size[i],
    markercolor = colors[i],            
    markeralpha = 0.7,                  
    markerstrokecolor = :black, 
    markerstrokewidth = 1.0,                 
    markerstrokealpha = 1.0,              
    linecolor =:black, 
    linewidth = 2,
)
end

# Annotation and limits
xpos = 2.7

annotate!(p0,
    xpos, r_tilde_GOE + 0.03,
    text(L"\mathrm{GOE}", 100, :darkorange)
)

annotate!(p0,
    xpos, r_tilde_poisson - 0.03,
    text(L"\mathrm{Poisson}", 100, :green)
)


ylims!(p0, - 0.02, r_tilde_GOE + 0.09)
xl, yl = xlims(p0), ylims(p0)

annotate!(p0,
    xl[1] + 0.14*(xl[2]-xl[1]),
    yl[1] + 0.93*(yl[2]-yl[1]),
    text("(a)", :right, 30)
)
display(p0)

Thermodynamic extrapolation of $\langle \tilde{\mathfrak{r}}\rangle$ as a function of $1/N$ for $\alpha {\in}[0,3]$ for the clean system in the $(m,k){=}(0,1)$ sector.

In [ ]:

pgfplotsx()

# Font sizes
guide_fs  = 5 * 7
tick_fs   = 5 * 20
legend_fs = 5 * 12
title_fs  = 5 * 15

# Parameters
N_list = [12, 14, 16,18,20,22]
#alpha_list = 0.0:0.2:3.0
alpha_list=vcat([0.0,0.01,0.05,0.1],0.2:0.2:3.0)
delta=0.0
hjj=0.0
mz = 0.0
inv_N_list = 1.0 ./ N_list

# Reference values
hline_values = [
    (r_tilde_poisson, :green, :dash, L"\mathrm{Poisson}"),
    (r_tilde_GOE,     :orange, :dot, L"\mathrm{GOE}"),
]

# Colormap for alpha
palet = collect(cgrad(:lightrainbow, length(alpha_list), categorical=true))

#Plot setup
            p1 = plot(
                xlabel = L"\fontsize{100}{40}\selectfont 1/N",
                ylabel = L"\fontsize{100}{40}\selectfont \langle \tilde{\mathfrak{r}} \rangle",
                title  = L"\fontsize{100}{38}\selectfont \mathrm{Clean\,limit}",
                framestyle = :box,
                size = (1000, 600),
                dpi = 300,
                grid = false,
                guidefontsize = guide_fs,
                tickfontsize = tick_fs,
                titlefontsize = title_fs,
                left_margin = 7mm,
                right_margin = 10mm,       
                bottom_margin = 7mm,
                fontfamily = "Computer Modern",
                xlims = (minimum(inv_N_list)-0.002, maximum(inv_N_list)+0.002),
            )

# Dupilicate plot to add the colorbar
scatter!(p1, [inv_N_list[1]], [r_tilde_poisson],
        marker_z = [0.0], 
        clims = (0.0, 3.0),
        color = :lightrainbow,
        markersize = 0,
        markeralpha = 0,
        label = "",
        colorbar = :right,
        colorbar_tickfontsize = 100, 
        colorbar_title = L"\alpha",
        colorbar_titlefontsize =100,
        colorbar_title_location=:bottom,
        right_margin = 10mm 
    )

            # Add horizontal reference lines
            for (y, col, style, lbl) in hline_values
                hline!(p1, [y]; color=col, linestyle=style, linewidth=3, alpha=1.0, label="")
            end

            # Plot curves for each alpha
            for (i,a) in enumerate(alpha_list)
                y_vals = [r_stat_HS_gen[(N, a, mz, delta, hjj)] for N in N_list]
                r_std_err_list = [r_stat_HS_gen_std_error[(N, a, mz, delta, hjj)] for N in N_list ]
                
                plot!(p1, inv_N_list, y_vals;
                    yerror = r_std_err_list,          
                    alpha = 0.7,                     
                    linewidth =  3.5,
                    color = palet[i],
                    label = "" )
            end

# Annotation and limits
xpos = 0.081

annotate!(p1,
    xpos, r_tilde_GOE + 0.025,
    text(L"\mathrm{GOE}", 100, :darkorange)
)

annotate!(p1,
    0.049, r_tilde_poisson - 0.035,
    text(L"\mathrm{Poisson}", 100, :green)
)

ylims!(p1, - 0.02, r_tilde_GOE + 0.09)

xl, yl = xlims(p1), ylims(p1)

annotate!(p1,
    xl[1] + 0.14*(xl[2]-xl[1]),
    yl[1] + 0.93*(yl[2]-yl[1]),
    text("(b)", :right, 30)
)

display(p1)

Number of unique energies in the Haldane Shastry model

In [ ]:
file="HS_num_of_unique_energies_N_10_to_52.jld2"
@load file num_unique

In [ ]:
pgfplotsx()

guide_fs  = 2 * 20
tick_fs   = 2 * 20
legend_fs = 2 * 12
title_fs  = 2 * 15

a=2.0
l=9.0:0.1:53
N_list=10:2:52
p2=scatter(N_list, num_unique;
    xlabel = L"N",
    ylabel = L"\mathcal{N}_E",
    title = L"\mathrm{Haldane{-}Shastry\,\, model}\, (\alpha{=} %$a)",
    label =L"\mathrm{Exact \,\, result}" ,
    size=(800, 600),
    legend=:bottomright,
    markersize=5,
    dpi = 300,
    guidefontsize = guide_fs,
    tickfontsize = tick_fs,
    legendfontsize = legend_fs,
    titlefontsize = title_fs,
    fontfamily = "Computer Modern",
    left_margin = 7mm,
    right_margin=5mm,
    bottom_margin = 7mm,
    grid = false,
    framestyle = :box,
    color = "#56B4E9",
    yscale=:log10,
    alpha=1.0,
)

unique_ub(N)=(N/12)*(N^2 +2 )+1

plot!(l,unique_ub.(l),
    color=:red,
    label=L"\mathrm{Upper \,\, bound}",
    linewidth=2.0)

xlims!(p2,9,53)

xl, yl = xlims(p2), ylims(p2)

annotate!(p2,
    xl[1] + 0.14*(xl[2]-xl[1]),
    yl[1] + 10^(0.96*(log10(yl[2]-yl[1])))  ,
    text("(c)", :right, 30))


display(p2)

All clean plots in single panel

In [ ]:

plots = Plots.Plot[]

# Plot a: N=16 Level statistics of clean system
push!(plots, p0)

# Plot b: Thermodynamic extrapolation
plot!(p1, right_margin = 23mm)
push!(plots, p1)

# Plot c: Degeneracy plot N=20
push!(plots, p2)

# ---- FINAL 1 × 3 GRID ----
finalplot = plot(
    plots...,
    layout = (1, 3),
    size = (2400, 600),
    margin = 8mm,
    bottom_margin = 10mm,
    top_margin = 10mm,
)

In [ ]:
savefig(finalplot,"figures_correction/Level_Stat_z_h_dis_HS_gen_clean_plots.pdf")

#### Normalized degenracy HS model

Normalized degeneracy of the full spectrum and for $N{=}20$ spins at the Haldane-Shastry point at $\alpha{=}2$.

In [ ]:
N=36

file = "HS_unique_energies_degen_N_36.jld2"
@load file eigen_vals degeneracies;

In [ ]:
pgfplotsx()

guide_fs  = 2 * 20
tick_fs   = 2 * 20
legend_fs = 2 * 12
title_fs  = 2 * 15

a=2.0

xvals = eigen_vals*((pi^2)/(24*N^2))
yvals = (degeneracies/(2^N))*10^3

p2 = scatter(xvals, yvals,                        
    xlabel = L"\mathrm{Energy\,\,eigenvalues}",
    ylabel = L"\mathrm{Normalized\,\,Degeneracy}\,\,\times 10^3 ",
    title = L"\mathrm{Haldane{-}Shastry\,\, model}\, (\alpha{=} %$a)",
    markercolor = :deepskyblue,
    legend = false,
    size = (800,600),
    dpi = 300,
    guidefontsize = guide_fs,
    tickfontsize = tick_fs,
    titlefontsize = title_fs,
    fontfamily = "Computer Modern",
    left_margin = 7mm,
    right_margin = 5mm,
    bottom_margin = 7mm,
    grid = false,
    framestyle = :box)


xl, yl = xlims(p2), ylims(p2)

# annotate!(p2,
#     xl[1] + 0.14*(xl[2]-xl[1]),
#     yl[1] + 0.93*(yl[2]-yl[1]),
#     text("(c)", :right, 30)
# )
annotate!(p2,
    xl[1] + 0.93*(xl[2]-xl[1]),
    yl[1] + 0.93*(yl[2]-yl[1]),
    text(L"N{=}%$N", :right, 30)
)

display(p2)

In [ ]:
savefig(p2,"figures/HS_model_degeneracy_N_$(N).pdf")

#### All hj

Disorder averaged gap ratio $\langle \tilde{\mathfrak{r}}\rangle$ of both podition and magnetic field disorder Hamiltonian as a function of interaction parameter $\alpha$ for different values of position disorder strengths $\delta$ and magnetic field disorder strengths $h/J^{\rm NN}_\alpha$ in the zero magnetization sector. For each value of $\alpha$, $\delta$, and $h/J^{\rm NN}_\alpha$, the results are averaged over $1000$ independent disorder realizations. The grey vertical line indicates the points $\alpha{=}0,1,2$. Results are shown for $N{=}16$ spins with (a) $h/J^{\rm NN}_\alpha{=}0.1$; (b) $h/J^{\rm NN}_\alpha{=}0.3$; (c) $h/J^{\rm NN}_\alpha{=}0.5$; (d) $h/J^{\rm NN}_\alpha{=}0.8$; (e) $h/J^{\rm NN}_\alpha{=}1.0$; (f) $h/J^{\rm NN}_\alpha{=}1.3$; (g) $h/J^{\rm NN}_\alpha{=}1.5$; (h) $h/J^{\rm NN}_\alpha{=}2.0$; (i) $h/J^{\rm NN}_\alpha{=}4.0$.

In [ ]:
file1="r_stat_HS_gen.jld2"
file2="r_stat_HS_gen_std_error.jld2"
@load file1 r_stat_HS_gen;
@load file2 r_stat_HS_gen_std_error;

In [ ]:
pgfplotsx()  

#Font Size
guide_fs  = 5 * 7
tick_fs   = 5 * 20
legend_fs = 5 * 12
title_fs  = 5 * 15

#Parameters
N = 16
mz = 0.0
hj_list = [0.1,0.3,0.5,0.8,1.0,1.3,1.5,2.0,4.0]
delta_list = [0.0,0.1,0.3,0.5,0.8,1.0]
alpha_zoom = [0.01,0.05,0.1]

# Markers and colors
markers = [:rect, :circle, :pentagon, :diamond, :utriangle, :dtriangle]
marker_size=[12,10,9,9,9,9]

colors = [
    "#E69F00",  # orange → square
    "#009E73",  # green → circle
     "#CC79A7",  # reddish purple → hexagon
    "#D55E00",  # vermillion → diamond
    "#F0E442",  # yellow → up-triangle
    "#56B4E9",  # sky blue → down-triangle
]

#Reference lines
hline_values = [
    (r_tilde_GOE,     :darkorange, :dot),
    (r_tilde_poisson, :green,  :dash),
]

vline_values = [
    (0.0, :gray, :dash),
    (1.0, :gray, :dash),
    (2.0, :gray, :dash),
]

# Subplots
plots = Plots.Plot[]
panel_labels = collect('a':'i')

#Looping over all magnetic fields
for (idx, hjj) in enumerate(hj_list)

    # legend position logic
    legend_pos = idx == length(hj_list) ? (0.57, 0.8) : (0.18, 0.35)

    p = plot(
        xlabel = L"\fontsize{100}{40}\selectfont \alpha",
        ylabel = L"\fontsize{100}{40}\selectfont \langle \tilde{\mathfrak{r}} \rangle",
        title  = L"h/J_\alpha^{\mathrm{NN}}{=}%$hjj",
        legend = legend_pos,
        legend_columns = 2,
        framestyle = :box,
        size = (800,600),
        dpi = 300,
        grid = false,
        guidefontsize = guide_fs,
        tickfontsize = tick_fs,
        legendfontsize = legend_fs,
        titlefontsize = title_fs,
        fontfamily = "Computer Modern",
    )

    # Plotting reference lines 
    for (y,c,s) in hline_values
        hline!(p, [y]; color=c, linestyle=s, label="", linewidth=3)
    end

    for (x,c,s) in vline_values
        vline!(p, [x]; color=c, linestyle=s, label="", linewidth=3)
    end

    # curves for each δ 
    for (i, delta) in enumerate(delta_list)

        if delta == 0.0
            alpha_list = vcat(alpha_zoom, 0.0:0.2:3.0)
        else
            alpha_list = vcat(alpha_zoom, 0.2:0.2:3.0)
        end

    # Mean ⟨r̃⟩ values
    r_mean_avg_list = [
        r_stat_HS_gen[(N, a, mz, delta, hjj)]
        for a in alpha_list
    ]

    # Corresponding standard errors
    r_std_err_list = [
        r_stat_HS_gen_std_error[(N, a, mz, delta, hjj)]
        for a in alpha_list
    ]

    # Scatter plot with vertical error bars

        scatter!(
            p,
            alpha_list,
            r_mean_avg_list;
            yerror = r_std_err_list,
            label = L"\delta {=} %$delta\,\,",
            marker = markers[i],
            markersize = marker_size[i],
            markercolor = colors[i],
            markeralpha = 0.7,
            markerstrokecolor = :black, 
            markerstrokewidth = 1.0,
            linecolor = colors[i],
            linewidth = 2,
        )
    end

# Annotation and limits
    xpos = 2.7

    annotate!(
        p,
        xpos, r_tilde_GOE + 0.01,
        text(L"\mathrm{GOE}", 100, :darkorange)
    )

    annotate!(
        p,
        xpos, r_tilde_poisson - 0.01,
        text(L"\mathrm{Poisson}", 100, :green)
    )

    ylims!(p, r_tilde_poisson - 0.02, r_tilde_GOE + 0.03)

    xl, yl = xlims(p), ylims(p)

annotate!(p,
    xl[1] + 0.14*(xl[2]-xl[1]),
    yl[1] + 0.93*(yl[2]-yl[1]),
    text("($(panel_labels[idx]))", :right, 30,  "Computer Modern")
)

    push!(plots, p)
end

# ---- FINAL 3 × 3 GRID ----
finalplot = plot(
    plots...,
    layout = (3,3),
    size = (2400,1800),
    margin = 8mm,
    right_margin = 10mm,
    bottom_margin = 10mm,
    top_margin = 10mm,
)

display(finalplot)

In [ ]:
savefig(finalplot,"figures/Level_Stat_z_h_dis_HS_gen_N_$(N)_hj_all.pdf")

In [ ]:
file1="r_stat_HS_gen_spin_1.jld2"
file2="r_stat_HS_gen_spin_1_std_error.jld2"

@load file1 r_stat_HS_gen_spin_1
@load file2 r_stat_HS_gen_spin_1_std_error

In [ ]:
pgfplotsx()  

#Font Size
guide_fs  = 5 * 7
tick_fs   = 5 * 20
legend_fs = 5 * 12
title_fs  = 5 * 15

#Parameters
N = 10
mz = 0.0
hj_list = [0.0,0.1,0.3,0.5,0.8,1.0,1.5,2.0,4.0]
delta_list = [0.0,0.1,0.3,0.5,0.8,1.0]
alpha_zoom = []#[0.01,0.05,0.1]

# Markers and colors
markers = [:rect, :circle, :pentagon, :diamond, :utriangle, :dtriangle]
marker_size=[12,10,9,9,9,9]

colors = [
    "#E69F00",  # orange → square
    "#009E73",  # green → circle
     "#CC79A7",  # reddish purple → hexagon
    "#D55E00",  # vermillion → diamond
    "#F0E442",  # yellow → up-triangle
    "#56B4E9",  # sky blue → down-triangle
]

#Reference lines
hline_values = [
    (r_tilde_GOE,     :darkorange, :dot),
    (r_tilde_poisson, :green,  :dash),
]

vline_values = [
    (0.0, :gray, :dash),
    (1.0, :gray, :dash),
    (2.0, :gray, :dash),
]

# Subplots
plots = Plots.Plot[]
panel_labels = collect('a':'i')

#Looping over all magnetic fields
for (idx, hjj) in enumerate(hj_list)

    # legend position logic
    if idx == 2
        legend_pos = (0.57, 0.35)
    elseif idx == length(hj_list)
        legend_pos = (0.18, 0.35) #(0.57, 0.8)
    else
        legend_pos = (0.18, 0.35)
    end

    p = plot(
        xlabel = L"\fontsize{100}{40}\selectfont \alpha",
        ylabel = L"\fontsize{100}{40}\selectfont \langle \tilde{\mathfrak{r}} \rangle",
        title  = L"h/J_\alpha^{\mathrm{NN}}{=}%$hjj",
        legend = legend_pos,
        legend_columns = 2,
        framestyle = :box,
        size = (800,600),
        dpi = 300,
        grid = false,
        guidefontsize = guide_fs,
        tickfontsize = tick_fs,
        legendfontsize = legend_fs,
        titlefontsize = title_fs,
        fontfamily = "Computer Modern",
    )

    # Plotting reference lines 
    for (y,c,s) in hline_values
        hline!(p, [y]; color=c, linestyle=s, label="", linewidth=3)
    end

    for (x,c,s) in vline_values
        vline!(p, [x]; color=c, linestyle=s, label="", linewidth=3)
    end

    # curves for each δ 
    for (i, delta) in enumerate(delta_list)

        if delta == 0.0
            alpha_list = vcat(alpha_zoom, 0.0:0.2:3.0)
        else
            alpha_list = vcat(alpha_zoom, 0.2:0.2:3.0)
        end

        if delta == 0.0 && hjj == 0.0
            N = 14
        else
            N = 10
        end

    # Mean ⟨r̃⟩ values
    r_mean_avg_list = [
        r_stat_HS_gen_spin_1[(N, a, mz, delta, hjj)]
        for a in alpha_list
    ]

    # Corresponding standard errors
    r_std_err_list = [
        r_stat_HS_gen_spin_1_std_error[(N, a, mz, delta, hjj)]
        for a in alpha_list
    ]

    # Scatter plot with vertical error bars

        scatter!(
            p,
            alpha_list,
            r_mean_avg_list;
            yerror = r_std_err_list,
            label = L"\delta {=} %$delta\,\,",
            marker = markers[i],
            markersize = marker_size[i],
            markercolor = colors[i],
            markeralpha = 0.7,
            markerstrokecolor = :black, 
            markerstrokewidth = 1.0,
            linecolor = colors[i],
            linewidth = 2,
        )
    end

# Annotation and limits
    xpos = 2.7



    if hjj== 0.0
        annotate!(p,
            xpos, r_tilde_GOE + 0.03,
            text(L"\mathrm{GOE}", 100, :darkorange)
        )

        annotate!(p,
            xpos, r_tilde_poisson - 0.05,
            text(L"\mathrm{Poisson}", 100, :green)
        )

        ylims!(p, - 0.02, r_tilde_GOE + 0.09)
    else
            annotate!(
                p,
                xpos, r_tilde_GOE + 0.01,
                text(L"\mathrm{GOE}", 100, :darkorange)
            )

            annotate!(
                p,
                xpos, r_tilde_poisson - 0.01,
                text(L"\mathrm{Poisson}", 100, :green)
            )
        ylims!(p, r_tilde_poisson - 0.02, r_tilde_GOE + 0.03)
    end

    if idx == 2
        legend_pos = (0.65,0.4) # (0.57, 0.35)
        annotate!(
                p,
                legend_pos,
                text(L"\mathrm{spin{-}1}", 100, :red)
            )
    else
        legend_pos = (0.26,0.4) #(0.18, 0.35)
        annotate!(
                p,
                legend_pos,
                text(L"\mathrm{spin{-}1}", 100, :red)
            )
    end

    # ylims!(p, r_tilde_poisson - 0.02, r_tilde_GOE + 0.03)

    xl, yl = xlims(p), ylims(p)

annotate!(p,
    xl[1] + 0.14*(xl[2]-xl[1]),
    yl[1] + 0.93*(yl[2]-yl[1]),
    text("($(panel_labels[idx]))", :right, 30,  "Computer Modern")
)

    push!(plots, p)
end

# ---- FINAL 3 × 3 GRID ----
finalplot = plot(
    plots...,
    layout = (3,3),
    size = (2400,1800),
    margin = 8mm,
    right_margin = 10mm,
    bottom_margin = 10mm,
    top_margin = 10mm,
)

display(finalplot)

In [ ]:
savefig(finalplot,"figures/Level_Stat_z_h_dis_HS_gen_N_$(N)_hj_all_spin_1.pdf")

#### Finite size Effects

Thermodynamic extrapolation of disorder averaged gap ratio $\langle \tilde{\mathfrak{r}}\rangle$ of position snd magnetic field disordered Hamiltonian as a function of $1/N$ for interaction parameter $\alpha{\in}(0,3]$, disorder strength $\delta$ and magnetic field strength $h/J^{\rm NN}_\alpha$ in the zero magnetization sector. For each value of $\alpha$, $\delta$, and $h/J^{\rm NN}_\alpha$, the results are averaged over $1000$ independent disorder realizations. Results are shown for (a) $\delta{=}0.1$ and $h/J^{\rm NN}_\alpha{=}0.0$; (b) $\delta{=}1.0$ and $h/J^{\rm NN}_\alpha{=}0.0$; (c) $\delta{=}0.0$ and $h/J^{\rm NN}_\alpha{=}0.5$; (d) $\delta{=}0.1$ and $h/J^{\rm NN}_\alpha{=}0.1$; (e) $\delta{=}0.5$ and $h/J^{\rm NN}_\alpha{=}0.5$; (f) $\delta{=}1.0$ and $h/J^{\rm NN}_\alpha{=}0.1$.

In [ ]:
file1="r_stat_HS_gen.jld2"
file2="r_stat_HS_gen_std_error.jld2"
@load file1 r_stat_HS_gen;
@load file2 r_stat_HS_gen_std_error;

In [ ]:
pgfplotsx()

# Font sizes
guide_fs  = 5 * 7
tick_fs   = 5 * 20
legend_fs = 5 * 12
title_fs  = 5 * 15

# Parameters
N_list = [10, 12, 14, 16]
alpha_list=vcat([0.0,0.01,0.05,0.1],0.2:0.2:3.0)
mz = 0.0
inv_N_list = 1.0 ./ N_list

# Reference values 
hline_values = [
    (r_tilde_poisson, :green,  :dash, L"\mathrm{Poisson}"),
    (r_tilde_GOE,     :orange, :dot,  L"\mathrm{GOE}")
]

# Colorscheme
palet = collect(cgrad(:lightrainbow, length(alpha_list), categorical = true))


# Panel order & labels
panel_params = [
    (0.0, 0.1),
    (0.0, 1.0),
    (0.1, 0.0),
    (0.1, 0.1),
    (0.5, 0.5),
    (0.1, 1.0)
]

panel_lims = [0.22,0.35,0.27,0.33,0.35,0.35]

panel_labels = ["(a)", "(b)", "(c)", "(d)", "(e)", "(f)"]

# Plots setup
function make_panel(delta, hjj, panel_label, panel_lims)

    p = plot(
        xlabel = L"\fontsize{100}{40}\selectfont 1/N",
        ylabel = L"\fontsize{100}{40}\selectfont \langle \tilde{\mathfrak{r}} \rangle",
        title  = L"\fontsize{100}{38}\selectfont \delta{=}%$delta , \,\, h/J^{\mathrm{NN}}_\alpha{=}%$hjj",
        framestyle = :box,
        grid = false,
        size = (800, 600),
        dpi = 300,
        guidefontsize = guide_fs,
        tickfontsize = tick_fs,
        legendfontsize = legend_fs,
        titlefontsize = title_fs,
        fontfamily = "Computer Modern",
        left_margin = 7mm,
        right_margin = 20mm,
        bottom_margin = 7mm,
        xlims = (minimum(inv_N_list)-0.002, maximum(inv_N_list)+0.002),
        
     )

    # Dummy plot for colorbars
   scatter!(p, [inv_N_list[1]], [panel_lims[1]],
        marker_z = [0.0], 
        clims = (0.0, 3.0),
        color = :lightrainbow,
        markersize = 0,
        markeralpha = 0,
        label = "",
        colorbar = :right,
        colorbar_tickfontsize = 100, 
        colorbar_title = L"\fontsize{100}{38}\selectfont \alpha",
        colorbar_titlefontsize = 100,
        colorbar_title_location=:bottom,
        right_margin = 50mm 
    )


    # reference lines
    for (y, col, style, _) in hline_values
        hline!(p, [y]; color = col, linestyle = style, linewidth = 3, label = "")
    end

    # data curves
    for (i, a) in enumerate(alpha_list)

        if a == 0.0 && delta != 0.0
            continue
        end

        y_vals = [
            r_stat_HS_gen[(N, a, mz, delta, hjj)]
            for N in N_list
        ]

        y_err = [
            r_stat_HS_gen_std_error[(N, a, mz, delta, hjj)]
            for N in N_list
        ]

        plot!(
            p,
            inv_N_list, y_vals;
            yerror = y_err,
            linewidth = 3.5,       
            color= palet[i],
            alpha = 0.7,
            label = ""
        )
    end

    # Annotations and limits
    annotate!(p, 0.095, r_tilde_GOE + 0.015,
        text(L"\mathrm{GOE}", 100, :darkorange)
    )

    annotate!(p, 0.065, r_tilde_poisson - 0.02,
        text(L"\mathrm{Poisson}", 100, :green)
    )

    ylims!(p, panel_lims, r_tilde_GOE + 0.04)

    xl, yl = xlims(p), ylims(p)

    annotate!(p,
        xl[1] + 0.14*(xl[2]-xl[1]),
        yl[1] + 0.93*(yl[2]-yl[1]),
        text(panel_label, :right, 30, "Computer Modern"))


    return p
end

# Build all panels
plots = Plots.Plot[]

for (i, (h,δ)) in enumerate(panel_params)
    push!(plots, make_panel(δ, h, panel_labels[i],panel_lims[i]))
end

# Final 2 × 3 grid
finalplot = plot(
    plots...,
    layout = (2, 3),
    size = (2500, 1200),
    margin = 8mm,
    right_margin = 30mm,
    bottom_margin = 10mm,
    top_margin = 10mm,
)

display(finalplot) 

In [ ]:
savefig(finalplot, "figures_correction/Level_Stat_finte_size.pdf")

In [ ]:
file1="r_stat_HS_gen_spin_1.jld2"
file2="r_stat_HS_gen_spin_1_std_error.jld2"

@load file1 r_stat_HS_gen_spin_1
@load file2 r_stat_HS_gen_spin_1_std_error

In [ ]:
pgfplotsx()

# Font sizes
guide_fs  = 5 * 7
tick_fs   = 5 * 20
legend_fs = 5 * 12
title_fs  = 5 * 15

# Parameters
N_list = [6,8,10]
alpha_list=vcat([0.0],0.2:0.2:3.0)
mz = 0.0
inv_N_list = 1.0 ./ N_list

# Reference values 
hline_values = [
    (r_tilde_poisson, :green,  :dash, L"\mathrm{Poisson}"),
    (r_tilde_GOE,     :orange, :dot,  L"\mathrm{GOE}")
]

# Colorscheme
palet = collect(cgrad(:lightrainbow, length(alpha_list), categorical = true))


# Panel order & labels
panel_params = [
    (0.0, 0.1),
    (0.0, 1.0),
    (0.1, 0.0),
    (0.1, 0.1),
    (0.5, 0.5),
    (0.1, 1.0)
]

panel_lims = [0.22,0.35,0.2,0.33,0.35,0.35]

panel_labels = ["(a)", "(b)", "(c)", "(d)", "(e)", "(f)"]

# Plots setup
function make_panel(delta, hjj, panel_label, panel_lims)

    p = plot(
        xlabel = L"\fontsize{100}{40}\selectfont 1/N",
        ylabel = L"\fontsize{100}{40}\selectfont \langle \tilde{\mathfrak{r}} \rangle",
        title  = L"\fontsize{100}{38}\selectfont \delta{=}%$delta , \,\, h/J^{\mathrm{NN}}_\alpha{=}%$hjj",
        framestyle = :box,
        grid = false,
        size = (800, 600),
        dpi = 300,
        guidefontsize = guide_fs,
        tickfontsize = tick_fs,
        legendfontsize = legend_fs,
        titlefontsize = title_fs,
        fontfamily = "Computer Modern",
        left_margin = 7mm,
        right_margin = 20mm,
        bottom_margin = 7mm,
        xlims = (minimum(inv_N_list)-0.002, maximum(inv_N_list)+0.002),
        
     )

    # Dummy plot for colorbars
   scatter!(p, [inv_N_list[1]], [panel_lims[1]],
        marker_z = [0.0], 
        clims = (0.0, 3.0),
        color = :lightrainbow,
        markersize = 0,
        markeralpha = 0,
        label = "",
        colorbar = :right,
        colorbar_tickfontsize = 100, 
        colorbar_title = L"\fontsize{100}{38}\selectfont \alpha",
        colorbar_titlefontsize = 100,
        colorbar_title_location=:bottom,
        right_margin = 50mm 
    )


    # reference lines
    for (y, col, style, _) in hline_values
        hline!(p, [y]; color = col, linestyle = style, linewidth = 3, label = "")
    end

    # data curves
    for (i, a) in enumerate(alpha_list)

        if a == 0.0 && delta != 0.0
            continue
        end

        y_vals = [
            r_stat_HS_gen_spin_1[(N, a, mz, delta, hjj)]
            for N in N_list
        ]

        y_err = [
            r_stat_HS_gen_spin_1_std_error[(N, a, mz, delta, hjj)]
            for N in N_list
        ]

        plot!(
            p,
            inv_N_list, y_vals;
            yerror = y_err,
            linewidth = 3.5,       
            color= palet[i],
            alpha = 0.7,
            label = ""
        )
    end

    # Annotations and limits
    annotate!(p, 0.135, r_tilde_GOE + 0.015,
        text(L"\mathrm{GOE}", 100, :darkorange)
    )

    annotate!(p, 0.13, r_tilde_poisson - 0.02,
        text(L"\mathrm{Poisson}", 100, :green)
    )

    ylims!(p, panel_lims, r_tilde_GOE + 0.04)

    xl, yl = xlims(p), ylims(p)

    annotate!(p,
        xl[1] + 0.14*(xl[2]-xl[1]),
        yl[1] + 0.93*(yl[2]-yl[1]),
        text(panel_label, :right, 30, "Computer Modern"))


    return p
end

# Build all panels
plots = Plots.Plot[]

for (i, (h,δ)) in enumerate(panel_params)
    push!(plots, make_panel(δ, h, panel_labels[i],panel_lims[i]))
end

# Final 2 × 3 grid
finalplot = plot(
    plots...,
    layout = (2, 3),
    size = (2500, 1200),
    margin = 8mm,
    right_margin = 30mm,
    bottom_margin = 10mm,
    top_margin = 10mm,
)

display(finalplot) 

In [ ]:
savefig(finalplot, "figures/Level_Stat_finte_size_spin_1.pdf")

#### Unique Energy statistics

Level statistics before [raw energies] and after removing the degeneracies [unique energies]; Mean gap ratio $\langle \tilde{\mathfrak{r}}\rangle$ as a function of $\alpha$ for the clean Hamiltonian in $(m,k){=}(0,1)$ symmetry sector and the full spectrum.

In [ ]:
file = "r_stat_unique_clean_HS_gen.jld2"
@load file r_stat_clean_HS_gen

r_uni_full = r_stat_clean_HS_gen[("full spectrum","unique")] ;
r_raw_sec = r_stat_clean_HS_gen[("k=1 sector","raw")] ;
r_uni_sec = r_stat_clean_HS_gen[("k=1 sector","unique")] ;

In [ ]:
pgfplotsx()

# Font sizes
guide_fs  = 5 * 7
tick_fs   = 5 * 20
legend_fs = 3 * 7
title_fs  = 5 * 15

# Parameters
N = 20
alpha_list = 0.0:0.2:3.0

# Horizontal reference values
hline_values = [
    (r_tilde_poisson, :green, :dash,  L"\mathrm{Poisson}"),
    (r_tilde_GOE,     :orange, :dot,  L"\mathrm{GOE}"),
]
vline_values = [
    (0.0, :gray, :dash ),
    (2.0, :gray, :dash),
]

# Colormap and markers 
palet = cgrad(:tab10)
c = [palet[i] for i in range(1,10)]

markers = [ :rect, :circle , :diamond]

            # Plot setup
            p = plot(
                xlabel = L"\fontsize{100}{40}\selectfont \alpha",
                ylabel = L"\fontsize{100}{40}\selectfont \langle \tilde{\mathfrak{r}} \rangle",
                title  = L"\fontsize{100}{38}\selectfont \mathrm{Clean\,limit}, \, N{=}%$N",
                legend =(0.01,0.3),  
                legend_font_halign = :left,
                framestyle = :box,
                size = (800, 600),
                dpi = 300,
                grid = false,
                guidefontsize = guide_fs,
                tickfontsize = tick_fs,
                legendfontsize = legend_fs,
                titlefontsize = title_fs,
                left_margin = 7mm,
                right_margin = 20mm,        
                bottom_margin = 7mm,
                fontfamily = "Computer Modern",
                ylims = (-0.02, 0.96),
            )

            # Add horizontal reference lines
            for (y, col, style, lbl) in hline_values
                hline!(p, [y]; color=col, linestyle=style, linewidth=3, alpha=1.0, label="")
            end

            for (y, col, style) in vline_values
                vline!(p, [y]; color=col, linestyle=style, label="", linewidth=4, alpha=0.5)
            end


            # Plot for mean ⟨r̃⟩ in sector (unique)
            
                scatter!(p, alpha_list, r_uni_sec;
                    linewidth = 2,
                    marker = markers[1],
                    markerstrokecolor = :black,
                    markersize = 8,
                    color = c[10],
                    label = L"\mathrm{Unique \,energies\,in\,} (m,k){=}(0,1) \mathrm{\, sector}" )
                 
            # Plot for mean ⟨r̃⟩ in sector (raw)
                scatter!(p, alpha_list, r_raw_sec;
                    linewidth = 2,
                    marker = markers[2],
                    markerstrokecolor =:black,
                    markersize = 6,
                    color = c[7],
                    label = L"\mathrm{Raw\, energies \,in\,} (m,k){=}(0,1) \mathrm{\, sector}" )

            # Plot for mean ⟨r̃⟩ in full spectrum (unique)
                scatter!(p, alpha_list, r_uni_full;
                    linewidth = 2,
                    marker = markers[3],
                    markerstrokecolor =:black,
                    markersize = 7,
                    color = c[2],
                    label = L"\mathrm{Unique\,energies\, in \,full \,spectrum}" )


#Annotations and limits
xpos = 2.7

annotate!(p,
    xpos, r_tilde_GOE + 0.04,
    text(L"\mathrm{GOE}", 100, :darkorange)
)

annotate!(p,
    xpos, r_tilde_poisson - 0.05, 
    text(L"\mathrm{Poisson}", 100, :green)
)

xl, yl = xlims(p), ylims(p)

annotate!(p,
    xl[1] + 0.14*(xl[2]-xl[1]),
    yl[1] + 0.93*(yl[2]-yl[1]),
    text("(a)", :right, 30,  "Computer Modern")
)

display(p)

Thermodynamic extrapolation of $\langle \tilde{\mathfrak{r}}\rangle$ as a function of $1/N$ for the Haldane-Shastry model ($\alpha{=}2.0$) of in $(m,k){=}(0,1)$ sector and the full spectrum.

In [ ]:
file= "r_stat_HS_N_10_to_50.jld2"

@load file r_stat_Haldane_Shastry_model

r_full_uni = r_stat_Haldane_Shastry_model[("full spectrum","unique")] ;
r_p1 = r_stat_Haldane_Shastry_model[("k=1 sector","raw")] ;
r_p1_unique = r_stat_Haldane_Shastry_model[("k=1 sector","unique")];



In [ ]:
pgfplotsx()
# Font sizes
guide_fs  = 5 * 7
tick_fs   = 5 * 20
legend_fs = 3 * 7
title_fs  = 5 * 15

# Parameters
N_list = 10:2:50

inv_N_list = 1.0 ./ N_list

# Horizontal reference values
hline_values = [
    (r_tilde_poisson, :green, :dash,  L"\mathrm{Poisson}"),
    (r_tilde_GOE,     :darkorange, :dot,  L"\mathrm{GOE}"),
    (r_tilde_GUE,:red,:dash,L"\mathrm{GUE}"),
    (r_tilde_GSE,:blue,:dot,L"\mathrm{GSE}"),
]

# Colormap for alpha
palet = cgrad(:tab10)

c = [palet[i] for i in range(1,10)]

markers = [ :rect, :circle , :diamond]

# Plot setup
            p2 = plot(
                xlabel = L"\fontsize{100}{40}\selectfont 1/N",
                ylabel = L"\fontsize{100}{40}\selectfont \langle \tilde{\mathfrak{r}} \rangle",
                title  = L"\fontsize{100}{38}\selectfont \mathrm{Haldane{-}Shastry\,model}\, (\alpha{=}2.0)",
                legend =(0.01,0.3), 
                legend_font_halign = :left,
                framestyle = :box,
                size = (800, 600),
                dpi = 300,
                grid = false,
                guidefontsize = guide_fs,
                tickfontsize = tick_fs,
                legendfontsize = legend_fs,
                titlefontsize = title_fs,
                left_margin = 7mm,
                right_margin = 20mm,       
                bottom_margin = 7mm,
                fontfamily = "Computer Modern",
                xlims = (minimum(inv_N_list)-0.002, maximum(inv_N_list)+0.002),
            )

            # Add horizontal reference lines
            for (y, col, style, lbl) in hline_values
                hline!(p2, [y]; color=col, linestyle=style, linewidth=3, alpha=1.0, label="")
            end

             # Plot for mean ⟨r̃⟩ in sector (unique)
                plot!(p2, inv_N_list, r_p1_unique;
                    linewidth = 2,
                    marker = markers[1],
                    markersize = 7,
                    color = c[10],
                    markerstrokecolor = :black,
                    label = L"\mathrm{Unique \, energies\, in\,} (m,k){=}(0,1) \mathrm{\, sector}" )
                 
                # Plot for mean ⟨r̃⟩ in sector (raw)
                plot!(p2, inv_N_list, r_p1;
                    linewidth = 2,
                    marker = markers[2],
                    markersize = 6,
                    color = c[7],
                    markerstrokecolor =:black,
                    label = L"\mathrm{Raw\, energies\,in\,} (m,k){=}(0,1) \mathrm{\, sector}" )

                 # Plot for mean ⟨r̃⟩ in full spectrum (unique)
                plot!(p2, inv_N_list, r_full_uni;
                    linewidth = 2,
                    marker = markers[3],
                    markersize = 7,
                    color = c[2],
                    markerstrokecolor =:black,
                    label = L"\mathrm{Unique\, energies\, in\,full\,spectrum}" )


#Annotations and limits
xpos = 0.092
xpos2= 0.033
annotate!(p2,
    xpos2, r_tilde_GOE - 0.04,
    text(L"\mathrm{GOE}", 100, :darkorange)
)

annotate!(p2,
    xpos, r_tilde_poisson - 0.05, 
    text(L"\mathrm{Poisson}", 100, :green)
)

annotate!(p2,
    xpos2, r_tilde_GUE +0.03, 
    text(L"\mathrm{GUE}", 100, :red)
)

annotate!(p2,
    xpos2, r_tilde_GSE + 0.05, 
    text(L"\mathrm{GSE}", 100, :blue)
)
           
ylims!(p2,-0.02,1.0)

xl, yl = xlims(p2), ylims(p2)

annotate!(p2,
    xl[1] + 0.93*(xl[2]-xl[1]),
    yl[1] + 0.93*(yl[2]-yl[1]),
    text("(b)", :right, 30,  "Computer Modern")
)

display(p2)

Thermodynamic extrapolation of $\langle \tilde{\mathfrak{r}}\rangle$ as a function of $1/N$ for the XXZ Hamiltonian at the anisotropic parameter $\Delta{=}\cos(\pi/3){=}0.5$ in $(m,k){=}(0,1)$ sector and the full spectrum.

In [ ]:
file="r_stat_XXZ_D_half.jld2"
@load file r_stat_XXZ_D_half

r_uni_full = r_stat_XXZ_D_half[("full spectrum","unique")] ;
r_raw_sec = r_stat_XXZ_D_half[("k=1 sector","raw")] ;
r_uni_sec = r_stat_XXZ_D_half[("k=1 sector","unique")] ;

In [ ]:
pgfplotsx()

# Font sizes
guide_fs  = 5 * 7
tick_fs   = 5 * 20
legend_fs = 3 * 7
title_fs  = 5 * 15

# Parameters
N_list = [10,12, 14, 16,18,20]
inv_N_list = 1.0 ./ N_list

# Horizontal reference values
hline_values = [
    (r_tilde_poisson, :green, :dash,  L"\mathrm{Poisson}"),
    (r_tilde_GOE,     :darkorange, :dot,  L"\mathrm{GOE}") ]

# Colormap and markers
palet = cgrad(:tab10)

c = [palet[i] for i in range(1,10)]

markers = [ :rect, :circle ,:diamond]

# plot setup
            p3 = plot(
                xlabel = L"\fontsize{100}{40}\selectfont 1/N",
                ylabel = L"\fontsize{100}{40}\selectfont \langle \tilde{\mathfrak{r}} \rangle",
                title  = L"\fontsize{100}{38}\selectfont \mathrm{XXZ\,model},\, \,\Delta{=}0.5",
                legend =:bottomright,   
                legend_font_halign = :left,
                framestyle = :box,
                size = (800, 600),
                dpi = 300,
                grid = false,
                guidefontsize = guide_fs,
                tickfontsize = tick_fs,
                legendfontsize = legend_fs,
                titlefontsize = title_fs,
                left_margin = 7mm,
                right_margin = 20mm,        
                bottom_margin = 7mm,
                fontfamily = "Computer Modern",
                xlims = (minimum(inv_N_list)-0.002, maximum(inv_N_list)+0.002),
            )

            # Add horizontal reference lines
            for (y, col, style, lbl) in hline_values
                hline!(p3, [y]; color=col, linestyle=style, linewidth=3, alpha=1.0, label="")
            end

            # Plot for mean ⟨r̃⟩ in sector (unique)
                plot!(p3, inv_N_list, r_uni_sec;
                    linewidth = 2,
                    marker = markers[1],
                    markersize = 8,
                    color = c[10],
                    markerstrokecolor = :black,
                    label = L"\mathrm{Unique \, energies\, in\,} (m,k){=}(0,1) \mathrm{\, sector}" )
                 
                # Plot for mean ⟨r̃⟩ in sector (raw)
                plot!(p3, inv_N_list, r_raw_sec;
                    linewidth = 2,
                    marker = markers[2],
                    markersize = 6,
                    color = c[7],
                    markerstrokecolor =:black,
                    label = L"\mathrm{Raw\, energies\,in\,} (m,k){=}(0,1) \mathrm{\, sector}" )

                # Plot for mean ⟨r̃⟩ in full spectrum (raw)
                plot!(p3, inv_N_list, r_uni_full;
                    linewidth = 2,
                    marker = markers[3],
                    markersize = 7,
                    color = c[2],
                    markerstrokecolor =:black,
                    label = L"\mathrm{Unique\, energies\,in\, full\,spectrum}" )

           
# annotations and limits
xpos = 0.095

annotate!(p3,
    xpos, r_tilde_GOE + 0.03,
    text(L"\mathrm{GOE}", 100, :darkorange)
)

annotate!(p3,
    xpos, r_tilde_poisson +0.04, 
    text(L"\mathrm{Poisson}", 100, :green)
)
           
ylims!(p3,-0.05, r_tilde_GOE+0.09 )

xl, yl = xlims(p3), ylims(p3)

annotate!(p3,
    xl[1] + 0.14*(xl[2]-xl[1]),
    yl[1] + 0.93*(yl[2]-yl[1]),
    text("(c)", :right, 30,  "Computer Modern")
)

display(p3)

Ploting these plots in a single pannel

In [ ]:
plots = Plots.Plot[]

#Clean limit
push!(plots, p)

#Haldane-Shastry Model
push!(plots, p2)

#XXZ Model
push!(plots, p3)

# ---- FINAL 1 × 3 GRID ----
finalplot = plot(
    plots...,
    layout = (1,3),
    size = (2200,600),
    margin = 8mm,
    right_margin = 10mm,
    bottom_margin = 10mm,
    top_margin = 10mm,
)
display(finalplot)

In [ ]:
savefig(finalplot,"figures/Level_Stat_unique.pdf")

#### Alpha Delta

Disorder averaged gap ratio $\langle \tilde{\mathfrak{r}}\rangle$ of both position and magnetic field disordered Hamiltonian as a function of effective rescaled parameter $\alpha\delta$ for different values of $\delta$ and  $h/J^{\rm NN}_\alpha$ in the zero magnetization sector.

In [ ]:
file1="r_stat_HS_gen_alpha_delta.jld2"
file2="r_stat_HS_gen_alpha_delta_std_error.jld2"
@load file1 r_stat_HS_gen_alpha_delta;
@load file2 r_stat_HS_gen_alpha_delta_std_error;

In [ ]:
pgfplotsx()


# Panel order & labels
panel_params = [
    (10,  "(a)" ),   
    (12, "(b)"  ),
    (14,  "(c)" ),
    (16,  "(d)" )
]

# Single-panel builder
function make_panel( N, panel_label)

    m = 0.0
    alpha_delta_list = 0.5:0.5:15.0
    delta_list = [0.1, 0.3, 0.5,0.8, 1.0]     
    hj=0.5

    hline_values = [
        (r_tilde_poisson, :green, :dash),
        (r_tilde_GOE,     :darkorange, :dot),
    ]

    # Font sizes
    guide_fs  = 5 * 7
    tick_fs   = 5 * 20
    legend_fs = 5 * 12
    title_fs  = 5 * 15


    # ------------------ Initialize plot ------------------
    p = plot(
        xlabel = L"\fontsize{100}{40}\selectfont \alpha \delta",
        ylabel = L"\fontsize{100}{40}\selectfont \langle \tilde{\mathfrak{r}} \rangle",
        title  = L"\fontsize{100}{38}\selectfont N{=}%$N,\,\, h/J_\alpha{=}%$hj",
        legend = (0.75,0.7),
        framestyle = :box,
        size = (800, 600),
        dpi = 300,
        grid = false,
        guidefontsize = guide_fs,
        tickfontsize = tick_fs,
        legendfontsize = legend_fs,
        titlefontsize = title_fs,
        left_margin = 7mm,
        right_margin = 5mm,
        bottom_margin = 7mm,
        fontfamily = "Computer Modern",
        xticks = collect(0:1:15),
    )

    # Add horizontal reference lines
    for (y, col, style) in hline_values
        hline!(p, [y]; color=col, linestyle=style, label="", linewidth=4, alpha=1.0)
    end

    markers = [ :circle, :pentagon, :diamond, :utriangle, :dtriangle]

    colors = [
        "#009E73",  # green 
        "#CC79A7",  # reddish purple 
        "#D55E00",  # vermillion 
        "#F0E442",  # yellow 
        "#56B4E9",  # sky blue 
    ]

    for (i, delta) in enumerate(delta_list)
        r_mean_avg_list = [
            r_stat_HS_gen_alpha_delta[(N, αδ, m, delta, hj)]
            for αδ in alpha_delta_list
        ]

        r_std_err_list = [
            r_stat_HS_gen_alpha_delta_std_error[(N, αδ, m, delta, hj)]
            for αδ in alpha_delta_list
        ]

        scatter!(
            p, alpha_delta_list, r_mean_avg_list;
            yerror = r_std_err_list,
            label = L"\delta {=} %$delta",
            marker = markers[i],
            markersize=8,
            markeralpha=0.8,
            color = colors[i],
            markerstrokecolor=:black,
            linewidth = 2,
        )
    end

    xpos = 13.5

    annotate!(p,
        xpos, r_tilde_GOE - 0.01,
        text(L"\mathrm{GOE}", 100, :orange)
    )

    annotate!(p,
        xpos, r_tilde_poisson +0.01,
        text(L"\mathrm{Poisson}", 100, :green)
    )

    ylims!(p, r_tilde_poisson - 0.04, r_tilde_GOE + 0.03)

    xl, yl = xlims(p), ylims(p)

    annotate!(p,
        xl[1] + 0.14*(xl[2]-xl[1]),
        yl[1] + 0.93*(yl[2]-yl[1]),
        text(panel_label, :right, 30)
    )

        return p
end


# All panels

plots = Plots.Plot[]

for (i, (N, panel_label)) in enumerate(panel_params)
    push!(plots, make_panel(N, panel_label))
end

finalplot = plot(
    plots...,
    layout = (1, 4),
    size = (3200, 600),
    margin = 8mm,
    right_margin = 10mm,
    bottom_margin = 10mm,
    top_margin = 10mm,
)

display(finalplot) 

In [ ]:
savefig(p,"figures/Level_Stat_z_h_dis_HS_gen_alpha_delta.pdf")

In [ ]:
pgfplotsx()


# Panel order & labels
panel_params = [
    (0.1,  "(a)" ),   
    (0.3, "(b)"  ),
    (0.8,  "(c)" ),
    (1.0,  "(d)" )
]

# Single-panel builder
function make_panel( hj, panel_label)

    N=16
    m = 0.0
    alpha_delta_list = 0.5:0.5:15.0
    delta_list = [0.1, 0.3, 0.5,0.8, 1.0]     
    

    hline_values = [
        (r_tilde_poisson, :green, :dash),
        (r_tilde_GOE,     :darkorange, :dot),
    ]

    # Font sizes
    guide_fs  = 5 * 7
    tick_fs   = 5 * 20
    legend_fs = 5 * 12
    title_fs  = 5 * 15


    # ------------------ Initialize plot ------------------
    p = plot(
        xlabel = L"\fontsize{100}{40}\selectfont \alpha \delta",
        ylabel = L"\fontsize{100}{40}\selectfont \langle \tilde{\mathfrak{r}} \rangle",
        title  = L"\fontsize{100}{38}\selectfont N{=}%$N,\,\, h/J_\alpha{=}%$hj",
        legend = (0.75,0.7),
        framestyle = :box,
        size = (800, 600),
        dpi = 300,
        grid = false,
        guidefontsize = guide_fs,
        tickfontsize = tick_fs,
        legendfontsize = legend_fs,
        titlefontsize = title_fs,
        left_margin = 7mm,
        right_margin = 5mm,
        bottom_margin = 7mm,
        fontfamily = "Computer Modern",
        xticks = collect(0:1:15),
    )

    # Add horizontal reference lines
    for (y, col, style) in hline_values
        hline!(p, [y]; color=col, linestyle=style, label="", linewidth=4, alpha=1.0)
    end

    markers = [ :circle, :pentagon, :diamond, :utriangle, :dtriangle]

    colors = [
        "#009E73",  # green 
        "#CC79A7",  # reddish purple 
        "#D55E00",  # vermillion 
        "#F0E442",  # yellow 
        "#56B4E9",  # sky blue 
    ]

    for (i, delta) in enumerate(delta_list)
        r_mean_avg_list = [
            r_stat_HS_gen_alpha_delta[(N, αδ, m, delta, hj)]
            for αδ in alpha_delta_list
        ]

        r_std_err_list = [
            r_stat_HS_gen_alpha_delta_std_error[(N, αδ, m, delta, hj)]
            for αδ in alpha_delta_list
        ]

        scatter!(
            p, alpha_delta_list, r_mean_avg_list;
            yerror = r_std_err_list,
            label = L"\delta {=} %$delta",
            marker = markers[i],
            markersize=8,
            markeralpha=0.8,
            color = colors[i],
            markerstrokecolor=:black,
            linewidth = 2,
        )
    end

    xpos = 13.5

    annotate!(p,
        xpos, r_tilde_GOE - 0.01,
        text(L"\mathrm{GOE}", 100, :orange)
    )

    annotate!(p,
        xpos, r_tilde_poisson +0.01,
        text(L"\mathrm{Poisson}", 100, :green)
    )

    ylims!(p, r_tilde_poisson - 0.04, r_tilde_GOE + 0.03)

    xl, yl = xlims(p), ylims(p)

    annotate!(p,
        xl[1] + 0.14*(xl[2]-xl[1]),
        yl[1] + 0.93*(yl[2]-yl[1]),
        text(panel_label, :right, 30)
    )

        return p
end


# All panels

plots = Plots.Plot[]

for (i, (N, panel_label)) in enumerate(panel_params)
    push!(plots, make_panel(N, panel_label))
end

finalplot = plot(
    plots...,
    layout = (1, 4),
    size = (3200, 600),
    margin = 8mm,
    right_margin = 10mm,
    bottom_margin = 10mm,
    top_margin = 10mm,
)

display(finalplot) 

In [ ]:
savefig(finalplot,"figures_correction/Level_Stat_z_h_dis_HS_gen_alpha_delta_various_hj.pdf")

#### Overlap analysis

Overlap of Jastrow wavefunction $|{\Psi^{\mathrm{J}}}\rangle$ of the form below with numerically evaluated ground state $|{\Psi_{\alpha}^{\mathrm{GS}}}\rangle$ at the point $\alpha$

$$
    |{\Psi^{\mathrm{J}}}\rangle = \sum_{\{\eta_1, \ldots, \eta_M\}} \psi^{\mathrm{J}}(\eta_1, \ldots, \eta_M) S_{\eta_1}^+ \cdots S_{\eta_M}^+ \,\, |{\underbrace{\downarrow \downarrow \cdots \downarrow \downarrow}_{\text{all } N \text{ spins } \downarrow} }\rangle.
$$

The sum is taken over all possible configurations of $M{=}N/2\,\,\uparrow $-spin coordinates $\eta_p$, and

$$
    \psi^{\mathrm{J}}(\eta_1, \ldots, \eta_M)=\prod^M_{p<q} (\eta_p\,-\,\eta_q)^2\,\prod^M_{p=1}\eta_p.
$$

where $\eta$'s are position of spins in the lattice.

In [ ]:
file1="Jastrow_overlap_with_HS_gen.jld2"
@load file1 Jastrow_overlap_with_HS_gen

In [ ]:
pgfplotsx()  # Make sure PGFPlotsX backend is active

# Font sizes
guide_fs  = 5 * 5
tick_fs   = 5 * 20
legend_fs = 5 * 12
title_fs  = 5 * 15

# Parameters
L_list = 12:2:22
delta = 0.0
alpha_list = vcat([0.0001, 0.001, 0.01], collect(0.1:0.1:3.0))

# Reference lines

vline_values = [
    (1.0,:grey,:dash),
    (2.0, :gray, :dash)
]

hline_values = [
    (1-(0.99)^2, :green, :solid)
]

# Main plot setup
p0 = plot(
    xlabel = L"\alpha",
    ylabel = L"1-|\langle \Psi^{\mathrm{J}}| \Psi_{\alpha}^{\mathrm{GS}}\rangle|^2",
    legend =:topright,
    legend_columns=2,
    legend_font_halign = :left,
    framestyle = :box,
    size = (800, 600),
    dpi = 300,
    grid = false,
    guidefontsize = guide_fs,
    tickfontsize = tick_fs,
    legendfontsize = legend_fs,
    titlefontsize = title_fs,
    left_margin = 7mm,
    right_margin = 5mm,
    bottom_margin = 7mm,
    fontfamily = "Computer Modern",
    xticks = collect(floor(Int, minimum(alpha_list)):ceil(Int, maximum(alpha_list))),
)

# Add vertical reference lines
for (x, col, style) in vline_values
    vline!(p0, [x]; color=col, linestyle=style, label="", linewidth=2, alpha=1.5)
end

for (y, col, style) in hline_values
    hline!(p0, [y]; color=col, linestyle=style, label="", linewidth=2, alpha=0.5)
end

# Markers and colors
markers = [:rect, :circle, :pentagon, :diamond, :utriangle, :dtriangle]
marker_size=[7,7,7,7,7,7]

colors = [
    "#E69F00",  # orange → square
    "#009E73",  # green → circle
     "#CC79A7",  # reddish purple → hexagon
    "#D55E00",  # vermillion → diamond
    "#F0E442",  # yellow → up-triangle
    "#56B4E9",  # sky blue → down-triangle
]

# Loop over delta values
for (i, L) in enumerate(L_list)
    
    # Mean ⟨r̃⟩ values
    overlap_list = [
        1-Jastrow_overlap_with_HS_gen[(L, a, delta)].mean_overlap
        for a in alpha_list
    ]

    # Scatter plot with vertical error bars
    scatter!(p0,alpha_list,overlap_list;
    label = L"N {=} %$L\,\,",
    marker = markers[i],
    markersize = marker_size[i],
    markercolor = colors[i],            
    markeralpha = 0.7,                   
    markerstrokecolor = :black, 
    markerstrokewidth = 1.0,                 
    markerstrokealpha = 1.0,              
    linecolor =:black, 
    linewidth = 2,
)
end

for (i, L) in enumerate(L_list)
    b = 1-Jastrow_overlap_with_HS_gen[(L, Inf, delta)].mean_overlap
    scatter!([3.2],[b],
            label = "",
            marker = markers[i],
            markersize = marker_size[i],
            markercolor = colors[i],            
            markeralpha = 0.7,                   
            markerstrokecolor = :red, 
            markerstrokewidth = 1.0,                 
            markerstrokealpha = 1.0,              
            linecolor =:black, 
            linewidth = 2,
        )
end
annotate!(p0, 2.8, 0.012, text(L"\mathrm{XXX\,\, model}",20,:red))
annotate!(p0, 2.8, 0.025, text(L"99% \,\mathrm{overlap}",20,:green))


xlims!(p0,-0.1,3.2)
ylims!(p0, 0.0,0.14)

display(p0)

In [ ]:
savefig(p0,"figures/Jastrow_overlap_clean_HS_gen.pdf")

In [ ]:
pgfplotsx()  # Make sure PGFPlotsX backend is active

# Font sizes
guide_fs  = 5 * 5
tick_fs   = 5 * 20
legend_fs = 5 * 12
title_fs  = 5 * 15

# Parameters
L_list = 12:2:22
delta = 0.0
alpha_list = vcat([0.0001, 0.001, 0.01], collect(0.1:0.1:3.0))

# Reference lines

vline_values = [
    (1.0,:grey,:dash),
    (2.0, :gray, :dash)
]

hline_values = [
    (log10((0.99)^2), :green, :solid)
]

# Main plot setup
p0 = plot(
    xlabel = L"\alpha",
    ylabel = L"log1|\langle \Psi^{\mathrm{J}}| \Psi_{\alpha}^{\mathrm{GS}}\rangle|^2",
    legend =:bottomright,
    legend_columns=2,
    legend_font_halign = :left,
    framestyle = :box,
    size = (800, 600),
    dpi = 300,
    grid = false
    guidefontsize = guide_fs,
    tickfontsize = tick_fs,
    legendfontsize = legend_fs,
    titlefontsize = title_fs,
    left_margin = 7mm,
    right_margin = 5mm,
    bottom_margin = 7mm,
    fontfamily = "Computer Modern",
    xticks = collect(floor(Int, minimum(alpha_list)):ceil(Int, maximum(alpha_list))),
)

# Add vertical reference lines
#for (x, col, style) in vline_values
    #vline!(p0, [x]; color=col, linestyle=style, label="", linewidth=2, alpha=1.5)
#end

for (y, col, style) in hline_values
    hline!(p0, [y]; color=col, linestyle=style, label="", linewidth=2, alpha=0.5)
end

# Markers and colors
markers = [:rect, :circle, :pentagon, :diamond, :utriangle, :dtriangle]
marker_size=[7,7,7,7,7,7]

colors = [
    "#E69F00",  # orange → square
    "#009E73",  # green → circle
     "#CC79A7",  # reddish purple → hexagon
    "#D55E00",  # vermillion → diamond
    "#F0E442",  # yellow → up-triangle
    "#56B4E9",  # sky blue → down-triangle
]

# Loop over delta values
for (i, L) in enumerate(L_list)
    
    # Mean ⟨r̃⟩ values
    overlap_list = [
        log10(Jastrow_overlap_with_HS_gen[(L, a, delta)].mean_overlap)
        for a in alpha_list
    ]

    # Scatter plot with vertical error bars
    scatter!(p0,alpha_list,overlap_list;
    label = L"N {=} %$L\,\,",
    marker = markers[i],
    markersize = marker_size[i],
    markercolor = colors[i],            
    markeralpha = 0.7,                   
    markerstrokecolor = :black, 
    markerstrokewidth = 1.0,                 
    markerstrokealpha = 1.0,              
    linecolor =:black, 
    linewidth = 2,
)
end

for (i, L) in enumerate(L_list)
    b = log10(Jastrow_overlap_with_HS_gen[(L, Inf, delta)].mean_overlap)
    scatter!([3.2],[b],
            label = "",
            marker = markers[i],
            markersize = marker_size[i],
            markercolor = colors[i],            
            markeralpha = 0.7,                   
            markerstrokecolor = :red, 
            markerstrokewidth = 1.0,                 
            markerstrokealpha = 1.0,              
            linecolor =:black, 
            linewidth = 2,
        )
end
#annotate!(p0, 2.8, 0.012, text(L"\mathrm{XXX\,\, model}",20,:red))
#annotate!(p0, 2.8, 0.025, text(L"99% \,\mathrm{overlap}",20,:green))


xlims!(p0,-0.1,3.2)
#ylims!(p0, 0.0,1.0)

display(p0)

In [ ]:
pgfplotsx()  # Make sure PGFPlotsX backend is active

# Font sizes
guide_fs  = 5 * 5
tick_fs   = 5 * 20
legend_fs = 5 * 12
title_fs  = 5 * 15

# Parameters
L_list = 6:2:16
delta_list = [0.1,0.5,1.0]
alpha_list = vcat([0.0,0.0001, 0.001, 0.01], collect(0.1:0.1:3.0))

# Reference lines

vline_values = [
    (1.0, :gray, :dash),
    (2.0, :gray, :dash),
]
lab=["(a)","(b)","(c)"]
legen = [:topright,:topright,:bottomright]
plots=[]
# Main plot setup
for (n,delta) in enumerate(delta_list)
p0 = plot(
    xlabel = L"\alpha",
    ylabel = L"1-|\langle \Psi^{\mathrm{J}}| \Psi_{\alpha}^{\mathrm{GS}}\rangle|^2",
    title = L"\delta {=} %$delta",
    legend =legen[n],
    legend_font_halign = :left,
    legend_columns=2,
    framestyle = :box,
    size = (800, 600),
    dpi = 300,
    grid = false,
    guidefontsize = guide_fs,
    tickfontsize = tick_fs,
    legendfontsize = legend_fs,
    titlefontsize = title_fs,
    left_margin = 7mm,
    right_margin = 5mm,
    bottom_margin = 7mm,
    fontfamily = "Computer Modern",
    xticks = collect(floor(Int, minimum(alpha_list)):ceil(Int, maximum(alpha_list))),
)

# Add vertical reference lines
for (x, col, style) in vline_values
   vline!(p0, [x]; color=col, linestyle=style, label="", linewidth=3, alpha=1.5)
end

# Markers and colors
markers = [:rect, :circle, :pentagon, :diamond, :utriangle, :dtriangle]
marker_size=[7,7,7,7,7,7]

colors = [
    "#E69F00",  # orange → square
    "#009E73",  # green → circle
     "#CC79A7",  # reddish purple → hexagon
    "#D55E00",  # vermillion → diamond
    "#F0E442",  # yellow → up-triangle
    "#56B4E9",  # sky blue → down-triangle
]

# Loop over delta values
for (i, L) in enumerate(L_list)
    
    # Mean ⟨r̃⟩ values
    overlap_list = [
        1-Jastrow_overlap_with_HS_gen[(L, a, delta)].mean_overlap
        for a in alpha_list
    ]

    # Corresponding standard errors
    SD_overlap_list = [
        Jastrow_overlap_with_HS_gen[(L, a, delta)].SD_overlap
        for a in alpha_list
    ]

    # Scatter plot with vertical error bars
    scatter!(p0,alpha_list,overlap_list;
    yerror = SD_overlap_list,
    label = L"N {=} %$L\,\,",
    marker = markers[i],
    markersize = marker_size[i],
    markercolor = colors[i],            
    markeralpha = 0.7,                   
    markerstrokecolor = :black, 
    markerstrokewidth = 1.0,                 
    markerstrokealpha = 1.0,              
    linecolor =colors[i], 
    linewidth = 2,
    linealpha = 0.7
)
end

ylims!(p0, 0.0,1.175)
xl, yl = xlims(p0), ylims(p0)

annotate!(p0,
    xl[1] + 0.14*(xl[2]-xl[1]),
    yl[1] + 0.93*(yl[2]-yl[1]),
    text(lab[n], :right, 30)
)
push!(plots,p0)
end
finalplot = plot(
    plots...,
    layout = (1, 3),
    size = (2400, 600),
    margin = 10mm,
    bottom_margin = 10mm,
    right_margin = 10mm,
    top_margin = 10mm,
)

In [ ]:
savefig(finalplot,"figures/Jastrow_overlap_disordered_HS_gen.pdf")

Overlap of numerically evaluated ground state of $\alpha{=}2$ with other $\alpha$ points.

In [ ]:
file1="HS_overlap_with_HS_gen.jld2"
@load file1 HS_overlap_with_HS_gen

In [ ]:
pgfplotsx()  # Make sure PGFPlotsX backend is active

# Font sizes
guide_fs  = 5 * 5
tick_fs   = 5 * 20
legend_fs = 5 * 12
title_fs  = 5 * 15

# Parameters
L_list = 6:2:16
delta_list = [0.1,0.5,1.0]
alpha_list = vcat([0.0,0.0001, 0.001, 0.01], collect(0.1:0.1:3.0))

# Reference lines

vline_values = [
    (1.0, :gray, :dash),
    (2.0, :gray, :dash),
]
hline_values = [
    (1-(0.98)^2, :green, :solid)
]

lab=["(a)","(b)","(c)"]
legen = [:topright,:topright,:topright]
plots=[]
# Main plot setup
for (n,delta) in enumerate(delta_list)
p0 = plot(
    xlabel = L"\alpha",
    ylabel = L"1-|\langle \Psi_{\alpha=2}^{\mathrm{GS}}| \Psi_{\alpha}^{\mathrm{GS}}\rangle|^2",
    title =  L"\delta {=} %$delta",
    legend =legen[n],
    legend_columns=2,
    legend_font_halign = :left,
    framestyle = :box,
    size = (800, 600),
    dpi = 300,
    grid = false,
    guidefontsize = guide_fs,
    tickfontsize = tick_fs,
    legendfontsize = legend_fs,
    titlefontsize = title_fs,
    left_margin = 7mm,
    right_margin = 5mm,
    bottom_margin = 7mm,
    fontfamily = "Computer Modern",
    xticks = collect(floor(Int, minimum(alpha_list)):ceil(Int, maximum(alpha_list))),
)

# Add vertical reference lines
for (x, col, style) in vline_values
    vline!(p0, [x]; color=col, linestyle=style, label="", linewidth=3, alpha=1.5)
end

for (y, col, style) in hline_values
    hline!(p0, [y]; color=col, linestyle=style, label="", linewidth=2, alpha=0.5)
end
# Markers and colors
markers = [:rect, :circle, :pentagon, :diamond, :utriangle, :dtriangle]
marker_size=[7,7,7,7,7,7]

colors = [
    "#E69F00",  # orange → square
    "#009E73",  # green → circle
     "#CC79A7",  # reddish purple → hexagon
    "#D55E00",  # vermillion → diamond
    "#F0E442",  # yellow → up-triangle
    "#56B4E9",  # sky blue → down-triangle
]

# Loop over delta values
for (i, L) in enumerate(L_list)
    
    # Mean ⟨r̃⟩ values
    overlap_list = [
        1-HS_overlap_with_HS_gen[(L, a, delta)].mean_overlap 
        for a in alpha_list
    ]

    # Corresponding standard errors
    SD_overlap_list = [
        HS_overlap_with_HS_gen[(L, a, delta)].SD_overlap 
        for a in alpha_list
    ]

    # Scatter plot with vertical error bars
    scatter!(p0,alpha_list,overlap_list;
    yerror = SD_overlap_list,
    label = L"N {=} %$L\,\,",
    marker = markers[i],
    markersize = marker_size[i],
    markercolor = colors[i],            
    markeralpha = 0.7,                   
    markerstrokecolor = :black, 
    markerstrokewidth = 1.0,                 
    markerstrokealpha = 1.0,              
    linecolor =colors[i], 
    linewidth = 2,
    linealpha = 0.7
)
end

ylims!(p0, 0.0,0.27)
xl, yl = xlims(p0), ylims(p0)

annotate!(p0,
    xl[1] + 0.14*(xl[2]-xl[1]),
    yl[1] + 0.93*(yl[2]-yl[1]),
    text(lab[n], :right, 30)
)

annotate!(p0, 2.7, 0.05, text(L"98% \,\mathrm{overlap}",20,:green))

push!(plots,p0)
end
finalplot = plot(
    plots...,
    layout = (1, 3),
    size = (2400, 600),
    margin = 10mm,
    bottom_margin = 10mm,
    right_margin = 10mm,
    top_margin = 10mm,
)

In [ ]:
savefig(finalplot,"figures/HS_overlap_disordered_HS_gen.pdf")

In [ ]:
file1 = "GS_overlap_HS_gen_spin_1.jld2"
file2 = "GS_overlap_HS_gen_std_spin_1.jld2"

@load file1 GS_overlap_HS_gen_spin_1
@load file2 GS_overlap_HS_gen_std_spin_1

In [ ]:
pgfplotsx()  # Make sure PGFPlotsX backend is active

# Font sizes
guide_fs  = 5 * 5
tick_fs   = 5 * 20
legend_fs = 5 * 12
title_fs  = 5 * 15

# Parameters
L_list = 6:2:16
delta = 0.0
alpha_list =0.1:0.1:3.0 # vcat([0.0001, 0.001, 0.01], collect(0.1:0.1:3.0))

# Reference lines

vline_values = [
    (1.0,:grey,:dash),
    (2.0, :gray, :dash)
]

hline_values = [
    (1-(0.99)^2, :green, :solid)
]

# Main plot setup
p0 = plot(
    xlabel = L"\alpha",
    ylabel = L"1-|\langle \Psi_{\alpha=2}^{\mathrm{GS}}| \Psi_{\alpha}^{\mathrm{GS}}\rangle|^2",
    title =  L"\delta {=} %$delta",
    legend =:topright,
    legend_columns=2,
    legend_font_halign = :left,
    framestyle = :box,
    size = (800, 600),
    dpi = 300,
    grid = false,
    guidefontsize = guide_fs,
    tickfontsize = tick_fs,
    legendfontsize = legend_fs,
    titlefontsize = title_fs,
    left_margin = 7mm,
    right_margin = 5mm,
    bottom_margin = 7mm,
    fontfamily = "Computer Modern",
    xticks = collect(floor(Int, minimum(alpha_list)):ceil(Int, maximum(alpha_list))),
)

# Add vertical reference lines
for (x, col, style) in vline_values
    vline!(p0, [x]; color=col, linestyle=style, label="", linewidth=2, alpha=1.5)
end

for (y, col, style) in hline_values
    hline!(p0, [y]; color=col, linestyle=style, label="", linewidth=2, alpha=0.5)
end

# Markers and colors
markers = [:rect, :circle, :pentagon, :diamond, :utriangle, :dtriangle]
marker_size=[7,7,7,7,7,7]

colors = [
    "#E69F00",  # orange → square
    "#009E73",  # green → circle
     "#CC79A7",  # reddish purple → hexagon
    "#D55E00",  # vermillion → diamond
    "#F0E442",  # yellow → up-triangle
    "#56B4E9",  # sky blue → down-triangle
]

# Loop over delta values
for (i, L) in enumerate(L_list)
    
    # Mean ⟨r̃⟩ values
    overlap_list = [
        1-GS_overlap_HS_gen_spin_1[(L, a, delta)]
        for a in alpha_list
    ]

    # Scatter plot with vertical error bars
    scatter!(p0,alpha_list,overlap_list;
    label = L"N {=} %$L\,\,",
    marker = markers[i],
    markersize = marker_size[i],
    markercolor = colors[i],            
    markeralpha = 0.7,                   
    markerstrokecolor = :black, 
    markerstrokewidth = 1.0,                 
    markerstrokealpha = 1.0,              
    linecolor =:black, 
    linewidth = 2,
)
end

for (i, L) in enumerate(L_list)
    b = 1- GS_overlap_HS_gen_spin_1[(L, Inf, delta)]
    scatter!([3.2],[b],
            label = "",
            marker = markers[i],
            markersize = marker_size[i],
            markercolor = colors[i],            
            markeralpha = 0.7,                   
            markerstrokecolor = :red, 
            markerstrokewidth = 1.0,                 
            markerstrokealpha = 1.0,              
            linecolor =:black, 
            linewidth = 2,
        )
end
annotate!(p0, 2.8, 0.012, text(L"\mathrm{XXX\,\, model}",20,:red))
annotate!(p0, 2.8, 0.025, text(L"99% \,\mathrm{overlap}",20,:green))
annotate!(p0, 2.8, 0.056, text(L"\mathrm{spin{-}1}",25,:orange))

xlims!(p0,-0.1,3.2)
ylims!(p0, 0.0,0.08)

xl, yl = xlims(p0), ylims(p0)

annotate!(p0,
    xl[1] + 0.15*(xl[2]-xl[1]),
    yl[1] + 0.93*(yl[2]-yl[1]),
    text("(a)", :right, 30))


plots=[]
push!(plots,p0)

In [ ]:
pgfplotsx()  # Make sure PGFPlotsX backend is active

# Font sizes
guide_fs  = 5 * 5
tick_fs   = 5 * 20
legend_fs = 5 * 12
title_fs  = 5 * 15

# Parameters
L_list = 6:2:12
delta_list = [0.1,0.5,1.0]
alpha_list = vcat([0.0], collect(0.1:0.1:3.0))

# Reference lines

vline_values = [
    (1.0, :gray, :dash),
    (2.0, :gray, :dash),
]
hline_values = [
    (1-(0.98)^2, :green, :solid)
]

lab=["(b)","(c)","(d)"]
legen = [:topright,:topright,:topright]

# Main plot setup
for (n,delta) in enumerate(delta_list)
p0 = plot(
    xlabel = L"\alpha",
    ylabel = L"1-|\langle \Psi_{\alpha=2}^{\mathrm{GS}}| \Psi_{\alpha}^{\mathrm{GS}}\rangle|^2",
    title =  L"\delta {=} %$delta",
    legend =legen[n],
    legend_columns=2,
    legend_font_halign = :left,
    framestyle = :box,
    size = (800, 600),
    dpi = 300,
    grid = false,
    guidefontsize = guide_fs,
    tickfontsize = tick_fs,
    legendfontsize = legend_fs,
    titlefontsize = title_fs,
    left_margin = 7mm,
    right_margin = 5mm,
    bottom_margin = 7mm,
    fontfamily = "Computer Modern",
    xticks = collect(floor(Int, minimum(alpha_list)):ceil(Int, maximum(alpha_list))),
)

# Add vertical reference lines
for (x, col, style) in vline_values
    vline!(p0, [x]; color=col, linestyle=style, label="", linewidth=3, alpha=1.5)
end

for (y, col, style) in hline_values
    hline!(p0, [y]; color=col, linestyle=style, label="", linewidth=2, alpha=0.5)
end
# Markers and colors
markers = [:rect, :circle, :pentagon, :diamond, :utriangle, :dtriangle]
marker_size=[7,7,7,7,7,7]

colors = [
    "#E69F00",  # orange → square
    "#009E73",  # green → circle
     "#CC79A7",  # reddish purple → hexagon
    "#D55E00",  # vermillion → diamond
    "#F0E442",  # yellow → up-triangle
    "#56B4E9",  # sky blue → down-triangle
]

# Loop over delta values
for (i, L) in enumerate(L_list)
    
    # Mean ⟨r̃⟩ values
    overlap_list = [
        1-GS_overlap_HS_gen_spin_1[(L, a, delta)]
        for a in alpha_list
    ]

    # Corresponding standard errors
    SD_overlap_list = [
        GS_overlap_HS_gen_std_spin_1[(L, a, delta)]
        for a in alpha_list
    ]

    # Scatter plot with vertical error bars
    scatter!(p0,alpha_list,overlap_list;
    yerror = SD_overlap_list,
    label = L"N {=} %$L\,\,",
    marker = markers[i],
    markersize = marker_size[i],
    markercolor = colors[i],            
    markeralpha = 0.7,                   
    markerstrokecolor = :black, 
    markerstrokewidth = 1.0,                 
    markerstrokealpha = 1.0,              
    linecolor =colors[i], 
    linewidth = 2,
    linealpha = 0.7
)
end

ylims!(p0, 0.0,0.27)
xl, yl = xlims(p0), ylims(p0)

if n==1
    annotate!(p0,
        xl[1] + 0.15*(xl[2]-xl[1]),
        yl[1] + 0.93*(yl[2]-yl[1]),
        text(lab[n], :right, 30)
    )

else
    annotate!(p0,
        xl[1] + 0.25*(xl[2]-xl[1]),
        yl[1] + 0.93*(yl[2]-yl[1]),
        text(lab[n], :right, 30)
    )
end

annotate!(p0, 2.5, 0.05, text(L"98% \,\mathrm{overlap}",20,:green))
annotate!(p0, 2.5, 0.21, text(L"\mathrm{spin{-}1}",25,:orange))
push!(plots,p0)
end

finalplot = plot(
    plots...,
    layout = (1, 4),
    size = (3200, 600),
    margin = 10mm,
    bottom_margin = 10mm,
    right_margin = 10mm,
    top_margin = 10mm,
)

In [ ]:
savefig(finalplot,"figures/overlap_HS_gen_spin_1_all.pdf")

#### Inverse Participation Ratio (IPR)

In [ ]:
@load "ipr_mid_10.jld2" ipr_mid_10_all_dict

In [ ]:
pgfplotsx()  

#Font Size
guide_fs  = 5 * 7
tick_fs   = 5 * 20
legend_fs = 5 * 12
title_fs  = 5 * 15

#Parameters
N = 14
mz = 0.0

hj_list = [0.0,0.1,0.3,0.5,0.8,1.0,1.5,2.0,4.0]
delta_list = [0.0,0.1,0.3,0.5,0.8,1.0]
alpha_zoom = []

# Markers and colors
markers = [:rect, :circle, :pentagon, :diamond, :utriangle, :dtriangle]
marker_size=[12,10,9,9,9,9]# - Degenracy

colors = [
    "#E69F00",  # orange → square
    "#009E73",  # green → circle
     "#CC79A7",  # reddish purple → hexagon
    "#D55E00",  # vermillion → diamond
    "#F0E442",  # yellow → up-triangle
    "#56B4E9",  # sky blue → down-triangle
]

#Reference lines
hline_values = [
    (-log2(1/binomial(14,7)), :red, :dot)#,
    #(r_tilde_poisson, :green,  :dash),
]

vline_values = [
    (0.0, :gray, :dash),
    (1.0, :gray, :dash),
    (2.0, :gray, :dash),
]

# Subplots
plots = Plots.Plot[]
panel_labels = collect('a':'i')

#Looping over all magnetic fields
for (idx, hjj) in enumerate(hj_list)

    # legend position logic
    legend_pos = idx == length(hj_list) ? (0.6,0.8) : :bottomleft
    p = plot(
        xlabel = L"\fontsize{100}{40}\selectfont \alpha",
        ylabel = L"\fontsize{100}{40}\selectfont -\mathrm{log_2(IPR)} ",
        title  = L"h/J_\alpha^{\mathrm{NN}}{=}%$hjj",
        legend = legend_pos,
        legend_columns = 2,
        framestyle = :box,
        size = (800,600),
        dpi = 300,
        grid = false,
        guidefontsize = guide_fs,
        tickfontsize = tick_fs,
        legendfontsize = legend_fs,
        titlefontsize = title_fs,
        fontfamily = "Computer Modern",
    )
    yli=[0.0,0.0]
    # # Plotting reference lines 
    for (y,c,s) in hline_values
        hline!(p, [y]; color=c, linestyle=s, label="", linewidth=3)
    end

    for (x,c,s) in vline_values
        vline!(p, [x]; color=c, linestyle=s, label="", linewidth=3)
    end

    # curves for each δ 
    for (i, delta) in enumerate(delta_list)
        if hjj ==0.0 && delta ==0.0
            continue
        end

        if delta == 0.0
            alpha_list = vcat(alpha_zoom, 0.0:0.2:3.0)
        else
            alpha_list = vcat(alpha_zoom, 0.2:0.2:3.0)
        end

    # Mean ⟨r̃⟩ values
    IPR_avg_list = [
        ipr_mid_10_all_dict[(N, mz,a, delta, hjj)].avg_ipr
        for a in alpha_list
    ]

    # Corresponding standard errors
    IPR_std_err_list = [
        ipr_mid_10_all_dict[(N, mz,a, delta, hjj)].SE_ipr
        for a in alpha_list
    ]

    # Scatter plot with vertical error bars

        scatter!(
            p,
            alpha_list,
            IPR_avg_list;
            yerror = IPR_std_err_list,
            label = L"\delta {=} %$delta\,\,",
            marker = markers[i],
            markersize = marker_size[i],
            markercolor = colors[i],
            markeralpha = 0.7,
            markerstrokecolor = :black, 
            markerstrokewidth = 1.0,
            linecolor = colors[i],
            linewidth = 2,
        )

        # yli[1]=minimum(IPR_avg_list)-0.5
        # yli[2]=maximum(IPR_avg_list) + 1
    end
# Annotation and limits
     xpos = 2.7
    annotate!(
        p,
        xpos, -log2(1/binomial(14,7)) - 0.6,
        text(L"\mathcal{D}(m{=}0)", 100, :red, "Computer Modern")
    )

    ylims!(p, 1.25, 12)

    xl, yl = xlims(p), ylims(p) 

annotate!(p,
    xl[1] + 0.14*(xl[2]-xl[1]),
    yl[1] + 0.9*(yl[2]-yl[1]),
    text("($(panel_labels[idx]))", :right, 30,  "Computer Modern")
)

    push!(plots, p)
end

# ---- FINAL 3 × 3 GRID ----
finalplot = plot(
    plots...,
    layout = (3,3),
    size = (2400,1800),
    margin = 8mm,
    right_margin = 10mm,
    bottom_margin = 10mm,
    top_margin = 10mm,
)

display(finalplot)

In [ ]:
savefig(finalplot, "figures/IPR_all_hj_log2_scale.pdf")

#### Linear Fit positional disorder

In [ ]:
using LinearAlgebra, Statistics

# parameters
N = 16
alpha_list = 0.2:0.2:3.0
delta_list = [0.1, 0.3, 0.5, 0.8, 1.0]
hjj = 0.0
mz = 0.0

# Linear fit y = a*x + b
function linear_fit(x, y)
    X = [ones(length(x)) x]
    β = X \ y
    return β[1], β[2]  # (intercept, slope)
end

# Standard error of the mean
sem(x) = std(x; corrected=true) / sqrt(length(x))

# Two different fitting starts
fit_starts = [1.0, 2.0]
fit_max = 3.0

α_vals = collect(alpha_list)

for fit_min in fit_starts
    slopes = Float64[]
    intercepts = Float64[]

    for delta in delta_list
        r_vals = [
            r_stat_HS_gen[(N, a, mz, delta, hjj)]
            for a in α_vals
        ]

        idx = (α_vals .>= fit_min) .& (α_vals .<= fit_max)
        α_fit = α_vals[idx]
        r_fit = r_vals[idx]

        intercept, slope = linear_fit(α_fit, r_fit)
        push!(slopes, slope)
        push!(intercepts, intercept)
        
    end

    println("Fit range α ∈ [$fit_min, $fit_max]")
    println("Slope:      mean = $(round(mean(slopes), digits=5)) ± $(round(sem(slopes), digits=5))")
    println("Intercept:  mean = $(round(mean(intercepts), digits=5)) ± $(round(sem(intercepts), digits=5))")
    println()
end

#### Interaction profile in lattice for large $\alpha$

In [ ]:
using Random
using LinearAlgebra
using Plots

pgfplotsx()

guide_fs  = 5 * 20
tick_fs   = 5 * 20
legend_fs = 5 * 12
title_fs  = 5 * 15

N = 16
R = 1.0
dot_size = 70
a = 100


deltas = [0.1, 1.0]


plt = plot(
    layout = (1, 2),
    size = (800, 400),
    dpi = 300,
    grid = false,
    framestyle=:none,
    guidefontsize = guide_fs,
    tickfontsize = tick_fs,
    titlefontsize = title_fs,
    left_margin = 0mm,
    right_margin = 5mm,
    bottom_margin = 2.5mm,
    fontfamily = "Computer Modern",
    legend = false
)

for (ax, δ) in enumerate(deltas)

    ind_shift= Vector(1:N) .+ δ*(rand(N) .- 0.5)  
    angles_rand = ((2*π) / N) .* ind_shift 

    x_rand = R .* cos.(angles_rand)
    y_rand = R .* sin.(angles_rand)

    scatter!(
        plt[ax],
        x_rand, y_rand,
        markersize = sqrt(dot_size),
        markercolor = :deepskyblue,
        markerstrokecolor = :black,
        aspect_ratio = :equal
    )
  
    d=[exp(im*(2*pi/N)*i) for i in ind_shift]
    for i in 1:N
        j = i == N ? 1 : i + 1
       
        r = abs(d[i] - d[j])
        if r > 0
            value = ((2π / N) / r)^a
            if value >0# 1e-10

                θ1 = ((2*π) / N) .* ind_shift[i] 

                if j==1
                    θ2 = (2*π)+((2*π) / N) .* ind_shift[j]
                else
                    θ2 = ((2*π) / N) .* ind_shift[j]
                end


                t = range(θ1,θ2;
                    length = 200
                )

                plot!(
                    plt[ax],
                    R .* cos.(t),
                    R .* sin.(t);
                    line_z=log10(value),
                    c = cgrad([:white, :orange,:red]),
                    lw = 6,
                    colorbar = :right,
                    colorbar_tickfontsize = 18,
                    colorbar_title = (ax==2) ? L"\log_{10}(\mathrm{\,Interaction\,\,strength\,})" : "",
                    colorbar_titlefontsize = 15,
                    
                    
                )
            end
        end
    end


    t = range(0, 2π; length = 400)
    plot!(
        plt[ax],
        cos.(t), sin.(t);
        color = :gray,
        lw = 1
    )


    plot!(
        plt[ax];
        title = L"\delta {=} %$δ",
        axis = false
    )

    
end


xl, yl = xlims(plt), ylims(plt)

annotate!(plt[2],
    xl[1] + 0.155*(xl[2]-xl[1]),
    yl[1] + 0.95*(yl[2]-yl[1]),
    text("(b)", :right,25))
    
annotate!(plt[1],
    xl[1] + 0.15*(xl[2]-xl[1]),
    yl[1] + 0.95*(yl[2]-yl[1]),
    text("(a)", :right, 25))
display(plt)

In [ ]:
savefig(plt,"figures/Lattice_a_100_N_16.pdf")